<a href="https://colab.research.google.com/github/RonakkudalAI/Practical-Machine-Learning/blob/main/Preprocessing_(Day_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


In [ ]:
data = {
    'Income': [50000, 60000, 55000, np.nan, 65000],
    'Gender': ['Male', 'Female', 'Female', 'Male', np.nan],
    'City': ['Mumbai', 'Delhi', 'Mumbai', 'Chennai', 'Delhi'],
    'Signup_Date': ['2022-01-01', '2021-06-15', '2022-03-10', '2020-12-20', '2021-08-05'],
    'Last_Login': ['2023-01-01', '2023-02-10', '2023-01-15', '2023-01-20', '2023-02-01'],
    'Purchased': [1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

In [ ]:
df

,Income,Gender,City,Signup_Date,Last_Login,Purchased
0,50000.0,Male,Mumbai,2022-01-01,2023-01-01,1
1,60000.0,Female,Delhi,2021-06-15,2023-02-10,0
2,55000.0,Female,Mumbai,2022-03-10,2023-01-15,1
3,NaN,Male,Chennai,2020-12-20,2023-01-20,0
4,65000.0,NaN,Delhi,2021-08-05,2023-02-01,1


In [ ]:
df['Signup_Date'] = pd.to_datetime(df["Signup_Date"])
df['Last_date'] = pd.to_datetime(df["Last_Login"])

df

,Income,Gender,City,Signup_Date,Last_Login,Purchased,Last_date
0,50000.0,Male,Mumbai,2022-01-01,2023-01-01,1,2023-01-01
1,60000.0,Female,Delhi,2021-06-15,2023-02-10,0,2023-02-10
2,55000.0,Female,Mumbai,2022-03-10,2023-01-15,1,2023-01-15
3,NaN,Male,Chennai,2020-12-20,2023-01-20,0,2023-01-20
4,65000.0,NaN,Delhi,2021-08-05,2023-02-01,1,2023-02-01


In [ ]:
#1 Handling Missing Value
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

In [ ]:
#Numerical Column
imputer = SimpleImputer(strategy = 'mean')
df['Income'] = imputer.fit_transform(df[['Income']])

In [ ]:
#Categorial Column
imputer_cat = SimpleImputer(strategy='most_frequent')
df['Gender'] = imputer_cat.fit_transform(df[['Gender']]).ravel()
df

,Income,Gender,City,Signup_Date,Last_Login,Purchased,Last_date
0,50000.0,Male,Mumbai,2022-01-01,2023-01-01,1,2023-01-01
1,60000.0,Female,Delhi,2021-06-15,2023-02-10,0,2023-02-10
2,55000.0,Female,Mumbai,2022-03-10,2023-01-15,1,2023-01-15
3,57500.0,Male,Chennai,2020-12-20,2023-01-20,0,2023-01-20
4,65000.0,Female,Delhi,2021-08-05,2023-02-01,1,2023-02-01


In [ ]:
#Data Feature Engineering
df['Signup_Year'] = df['Signup_Date'].dt.year
#Data Feature Engineering
df['Signup_Month'] = df['Signup_Date'].dt.month
df['Tenure_Days'] = (df['Last_date']-df['Signup_Date']).dt.days

In [ ]:
#One Hot Encoding
encoder = OneHotEncoder()
city = encoder.fit_transform(df[['City']])

city_df = pd.DataFrame(
    city.toarray(),
    columns=encoder.get_feature_names_out(['City'])
    )
df = pd.concat([df,city_df],axis = 1)

df


,Income,Gender,City,Signup_Date,Last_Login,Purchased,Last_date,Signup_Year,Signup_Month,Tenure_Days,City_Chennai,City_Delhi,City_Mumbai
0,50000.0,Male,Mumbai,2022-01-01,2023-01-01,1,2023-01-01,2022,1,365,0.0,0.0,1.0
1,60000.0,Female,Delhi,2021-06-15,2023-02-10,0,2023-02-10,2021,6,605,0.0,1.0,0.0
2,55000.0,Female,Mumbai,2022-03-10,2023-01-15,1,2023-01-15,2022,3,311,0.0,0.0,1.0
3,57500.0,Male,Chennai,2020-12-20,2023-01-20,0,2023-01-20,2020,12,761,1.0,0.0,0.0
4,65000.0,Female,Delhi,2021-08-05,2023-02-01,1,2023-02-01,2021,8,545,0.0,1.0,0.0


In [ ]:
# Feature Scaling
scaler = StandardScaler()
df[['Income']] = scaler.fit_transform(df[['Income']])

In [ ]:
#Feature Transform
df['Income_log'] = np.log(df['Income'])


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [ ]:
num_col = ['Income']
cat_col = ['Gender','City']


In [ ]:
n_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  # Handling missing values
    ('scaler', StandardScaler())                  # Feature scaling
])

# Categorical pipeline
c_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])

# Column Transformer
preprocessor = ColumnTransformer([
    ('num', n_pipeline, num_col),
    ('cat', c_pipeline, cat_col)
])

pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LogisticRegression)
])